2026-03-29 16:06:59,971 | httpx | HTTP Request: GET http://localhost:8080/v1/.well-known/openid-configuration "HTTP/1.1 404 Not Found"
2026-03-29 16:06:59,994 | httpx | HTTP Request: GET http://localhost:8080/v1/meta "HTTP/1.1 200 OK"
2026-03-29 16:07:00,095 | httpx | HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"
2026-03-29 16:07:00,120 | __main__ | Ingesting 1 / 3
2026-03-29 16:07:01,020 | __main__ | hashing for file 1 is 47d6cf6a382c18b449983593e1fa84bb
2026-03-29 16:07:01,306 | __main__ | File is already Ingested, Skipping Igestion for file 1
2026-03-29 16:07:01,306 | __main__ | Ingesting 2 / 3
2026-03-29 16:07:01,346 | __main__ | hashing for file 2 is f4b40cd1a67a0f4df758a95c91ffbf63
2026-03-29 16:07:01,464 | __main__ | File is already Ingested, Skipping Igestion for file 2
2026-03-29 16:07:01,464 | __main__ | Ingesting 3 / 3
2026-03-29 16:07:02,217 | __main__ | hashing for file 3 is 5aaad1664dd530b717be1f75a3911a02
2026-03-29 16:07:02,342 | __main_

0

In [ ]:
from collections import defaultdict

def get_documents():
    client=WEAVIATE_CLIENT()
    collection = client.collections.get(settings.COLLECTION)

    results = collection.query.fetch_objects(limit=1000)

    print(results.objects[1])
    client.close()
    
get_documents()

2026-03-29 15:50:07,642 | httpx | HTTP Request: GET http://localhost:8080/v1/.well-known/openid-configuration "HTTP/1.1 404 Not Found"
2026-03-29 15:50:07,664 | httpx | HTTP Request: GET http://localhost:8080/v1/meta "HTTP/1.1 200 OK"
2026-03-29 15:50:07,745 | httpx | HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"


Object(uuid=_WeaviateUUIDInt('00010c12-3e60-47aa-84ce-16fd3724ec1c'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'source': '/Users/rahul/Projects/Learning_Projects/Data/Master Circular for Credit Rating Agencies.pdf', 'chunk_number': 300.0, 'text': 'the issuers.\n30.2.2. A CRA shall disclose, in case of accepted ratings, its conflict of interest, if any,\nincluding the details of relationship – commercial or otherwise – between the\nissuer whose securities are being rated / any of its associate of such issuer and\nthe CRA or its subsidiaries.\n30.3.  Unsolicited credit ratings: While publishing unsolicited ratings and their\nmovements, a CRA apart from following all the applicable requirements in case of', 'is_active': 'unknown', 'total_pages': 109.0, 'hashing': 'unknown', 'total_chunks': 496.0, 'pages_number': 57.0, 'title': ''}, references=None, v

In [2]:
from app.sevices.weaviate_Manager import WEAVIATE_CLIENT
from app.core.settings import settings
results =WEAVIATE_CLIENT().collections.get(settings.COLLECTION).aggregate.over_all(
    group_by="hashing"
)

for group in results.groups:
    print(group.grouped_by.value)

unknown
47d6cf6a382c18b449983593e1fa84bb
5aaad1664dd530b717be1f75a3911a02
2daf088433017c1533f584b0ee5aa27f
b3d45509dc90a47f1723af7371573e4b
f4b40cd1a67a0f4df758a95c91ffbf63


In [5]:
from app.core.logger import get_logger
import numpy as np
logger = get_logger(__name__)
def mmr(doc_vector,query_vector,top_k=5,lambda_mul=0.5):
    #step 1 convert doc vec and query vec to array
    
    q = np.array(query_vector)
    d = np.array(doc_vector)
    
    #step 2 normalize
    q_normal = q/np.linalg.norm(q)
    d_normal = d/np.linalg.norm(d,axis=1,keepdims=True)
    
    #step 3 find relavance
    
    relavance = d_normal @ q_normal

    candidates = list(range(len(d)))
    selected =[]
    for _ in range(top_k):
        if not selected:
            best = max(candidates, key=lambda x: relavance[x])
        else:
            selected_vec = d_normal[selected]
            best = max(candidates, key=lambda x: lambda_mul *relavance[x] - (1 - lambda_mul)* np.max(d_normal[x] @ selected_vec.T))
        selected.append(best)
        candidates.remove(best)
    
    
    return selected
        

In [ ]:
from sentence_transformers import CrossEncoder
_cross_encoder = CrossEncoder("cross-encoder/ms-marco-electra-base")
def rerank(query,mmr_docs,top_n=3):
    pairs = [(query,doc.properties["text"]) for doc in mmr_docs]
    scores = _cross_encoder.predict(pairs)
    ranked = sorted(zip(scores,mmr_docs), key=lambda x: x[0],reverse=True)
    return [doc for _, doc in ranked[:top_n]]
    

In [26]:
from sentence_transformers import CrossEncoder
from app.core.settings import settings
_cross_encoder = CrossEncoder(settings.ENCODER)


    

2026-03-30 10:39:35,358 | httpx | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-electra-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-30 10:39:35,602 | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-electra-base/8b9301757513f8a695920c334c3a9f365ebeb357/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-electra-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
electra.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-30 10:39:36,006 | httpx | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-electra-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-30 10:39:36,087 | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-electra-base/8b9301757513f8a695920c334c3a9f365ebeb357/config.json "HTTP/1.1 200 OK"
2026-03-30 10:39:36,420 | httpx | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-electra-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-30 10:39:36,448 | httpx | HTTP Request: HEAD https://huggingface.co/api

In [34]:
def encoder(query,mmr_docs,top_n):
    pair = [(query,doc.properties["text"]) for doc in mmr_docs]
    print("paired query and docs",pair)
    score = _cross_encoder.predict(pair)
    print("predict_score", score)
    rerank = sorted(zip(score,mmr_docs),key=lambda x :x[0], reverse=True)
    print("rerank",rerank)
    return [doc for _,doc in rerank[:top_n]]

In [35]:
from app.core.settings import settings
from app.sevices.get_models import OLLAMA_EMBEDDING
from app.sevices.weaviate_Manager import WEAVIATE_CLIENT
from langchain_core.documents import Document
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
client= WEAVIATE_CLIENT()
embbeding_model = OLLAMA_EMBEDDING()
query="What is SEBI and What it is take care of?"
results = client.collections.get(settings.COLLECTION).query.hybrid(
    limit=20,
    query=query,
    vector=embbeding_model.embed_query("What is SEBI and What it is take care of?"),
    return_properties=["text","source","title","chunk_number", "total_chunks","pages_number","total_pages"],
    include_vector=True
)
docs = results.objects
doc_vector = [doc.vector["default"] for doc in docs]
query_vector= embbeding_model.embed_query("What is SEBI and What it is take care of?")
indexes = mmr(doc_vector=doc_vector, query_vector=query_vector)
mmr_docs = [docs[index] for index in indexes]
encoder(query,mmr_docs,top_n=3)

2026-03-30 10:43:39,830 | httpx | HTTP Request: GET http://localhost:8080/v1/.well-known/openid-configuration "HTTP/1.1 404 Not Found"
2026-03-30 10:43:39,866 | httpx | HTTP Request: GET http://localhost:8080/v1/meta "HTTP/1.1 200 OK"
2026-03-30 10:43:39,967 | httpx | HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"
/Users/rahul/Projects/Learning_Projects/.venv/lib/python3.11/site-packages/weaviate/warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
/var/folders/nx/v_cbhyj10r3_f035r0jncg6r0000gn/T/ipykernel_81820/3000438279.py:7: ResourceWarning: unclosed <socket.socket fd=108, family=30, type=1, proto=6, laddr=('::1', 61083, 0, 0), raddr=('::1', 8080, 0, 0)>
  client= WEAVIATE_CLIENT()
2026-03-30 10:43:40,268 | httpx | HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
202

paired query and docs [('What is SEBI and What it is take care of?', 'confidentiality of information.\n\uf0b7\nTo do a proper and unbiased risk – profiling and suitability assessment of\nthe client.\n\uf0b7\nTo conduct audit annually.\n\uf0b7\nTo disclose the status of complaints on its website.\n\uf0b7\nTo disclose the name, proprietor name, type of registration, registration\nnumber, validity, complete address with telephone numbers and associated\nSEBI Office details (i.e. Head office/ regional/ local Office) on its website.\n\uf0b7'), ('What is SEBI and What it is take care of?', 'to evaluate the application.\n9. At the “Evaluation Stage”, SEBI shall work with the applicant to determine the specific\nregulatory requirements and conditions (including test parameters and control'), ('What is SEBI and What it is take care of?', 'provisions of the SEBI Regulations and Circulars issued thereunder.\n******'), ('What is SEBI and What it is take care of?', 'complaint with SEBI on SEBI’s po

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

predict_score [0.11589272 0.2744549  0.00242534 0.88931406 0.13743715]
rerank [(np.float32(0.88931406), Object(uuid=_WeaviateUUIDInt('82dfce7d-b1c9-454b-b2ee-5f0d0740e701'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'title': '', 'text': 'complaint with SEBI on SEBI’s portal, named, “SCORES”, which is a centralized\nweb-based complaints redress system. SEBI takes up the complaints registered\nvia SCORES (https://scores.gov.in/scores/Welcome.html) with the concerned\nREIT / intermediary for timely redressal. SCORES facilitates tracking the status of', 'total_chunks': 1007.0, 'source': '/Users/rahul/Projects/Learning_Projects/Data/Master Circular for Real Estate Investment Trusts (REITs) .pdf', 'pages_number': 207.0, 'chunk_number': 972.0, 'total_pages': 216.0}, references=None, vector={'default': [-0.06845908612012863, -0.01627439819276333, -0.007507

[Object(uuid=_WeaviateUUIDInt('82dfce7d-b1c9-454b-b2ee-5f0d0740e701'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'title': '', 'text': 'complaint with SEBI on SEBI’s portal, named, “SCORES”, which is a centralized\nweb-based complaints redress system. SEBI takes up the complaints registered\nvia SCORES (https://scores.gov.in/scores/Welcome.html) with the concerned\nREIT / intermediary for timely redressal. SCORES facilitates tracking the status of', 'total_chunks': 1007.0, 'source': '/Users/rahul/Projects/Learning_Projects/Data/Master Circular for Real Estate Investment Trusts (REITs) .pdf', 'pages_number': 207.0, 'chunk_number': 972.0, 'total_pages': 216.0}, references=None, vector={'default': [-0.06845908612012863, -0.01627439819276333, -0.007507652975618839, 0.010648159310221672, -0.003351463470607996, 0.0058434465900063515, 0.016133515164256096,

In [ ]:
from typing import TypedDict

class RAGstate(TypedDict):
    question : str
    context: str
    answer : str
    faithfullness : float
    answer_relavancy : float
    retry_count : int
    session_id: str
    
state = RAGstate()



'rahul'

In [ ]:
from langgraph.graph import StateGraph, END
from app.graph.nodes import generate,query_rewritter,retriever
from app.graph.state import RAGstate
def build_graph():
    graph = StateGraph(RAGstate)
    
    graph.add_node("retriever",retriever.retriever)
    graph.add_node("generator",generate.generator)
    graph.add_node("re_writter",query_rewritter.query_rewritter)
    
    
    graph.set_entry_point("retriever")
    graph.add_edge("retriever","generator")
    graph.add_edge("generator",END)
    
    return graph.compile()

initial_state: RAGstate = {
    "question": "question",
    "context": "",
    "answer": "",
    "faithfullness": 0.0,
    "answer_relavancy": 0.0,
    "retry_count": 0,
    "session_id": "2332"
}
graph = build_graph()




In [2]:
graph.invoke(initial_state)

{'question': 'question',
 'context': '',
 'answer': '',
 'faithfullness': 100.0,
 'answer_relavancy': 0.0,
 'retry_count': 0,
 'session_id': '2332'}